# <span style="color:green"> Script for segmentation in slicer and definition of cylindrical 3D ROI

## <span style="color:black"> Black: stages in code
## <span style="color:red"> RED = ACTION REQUIRED
## <span style="color:Silver"> Silver = data resulting from action (also can be changed)
## <span style="color:blue"> Blue = notes

## <span style="color:blue"> make sure the CT scans are from the side so that sagital would be top (coronal = frontal)

# Load libraries

In [1]:
from pathlib import Path
import slicer
import vtk
import pyvista as pv
import math
import numpy as np
import os

# <span style="color:red"> User input

## .nrrd file path

In [2]:
import os

# Get the current working directory
current_directory = os.getcwd()

print("Current working directory:", current_directory)


Current working directory: /config/Downloads/python_modeling


In [3]:
# Specify the directory of microCT files (first .jpg file)
nrrd_file = r'tail1/CEP1/microCT_scan.nrrd'

# Preparing directories

In [4]:
# Specify the parent path
path = Path(nrrd_file)
parentpath = path.parent

# Prepare 3dSlicer and Import CT data

## Import .nrrd file

In [5]:
# Load the NRRD file
volume_node = slicer.util.loadVolume(nrrd_file)

## Build cylinder model parameters on xz plane ←↑ (GA plane)

<span style="color:red"> ACTION REQUIRED: Draw points in Slicer UI (The points can be edited and changed after)
    

<span style="color:blue">Before running the code below, go to the Slicer UI and draw a line in the ```A``` plane (i.e. ```xz``` plane). 

<span style="color:blue">Set the name of the line node to ```L```. Any name is actually fine, but you should change the <span style="color:red">ACTION</span> input cell accordingly (```name_lineNode```)

In [6]:
# HERE the ROI is the same as model but a bit smaller to ensure no outsides are taken
cylinder_height = 1 #mm 
cylinder_radius = 0.5 #mm 

# Create ROI Cylinder of center at center of "L" and direction per. to "L"

## Get the lines "L", "L_1", "L_2" that represent left, mid, and right lines respectively

In [7]:
left_line = slicer.util.getNode("L")
mid_line = slicer.util.getNode("L_1")
right_line = slicer.util.getNode("L_2")

## Find the center and direction of left cylinder  

In [8]:
# Get points of the line
left_point1 = vtk.vtkVector3d(0,0,0)
left_line.GetNthControlPointPosition(0,left_point1)

left_point2 = vtk.vtkVector3d(0,0,0)
left_line.GetNthControlPointPosition(1,left_point2)

# arc tangent of in radians
angle = math.atan2(left_point2[2] - left_point1[2], left_point2[0] - left_point1[0])

# Center of cylinder
left_center_cylinder = [left_point1[0] + cylinder_radius * math.cos(angle) + np.sign(left_point1[2] - left_point2[2]) * np.sign(math.cos(angle)) * cylinder_height/2 * abs(math.sin(angle)),
                   left_point1[1],
                   left_point1[2] + cylinder_radius * math.sin(angle) + np.sign(math.cos(angle)) * cylinder_height/2 * math.cos(angle)]

# direction of cylinder
left_cylinder_direction = [np.sign(left_point1[2] - left_point2[2]) * np.sign(math.cos(angle)) * abs(math.sin(angle)), 
                      0,
                      abs(math.cos(angle))]

## Find the center and direction of mid cylinder  

In [9]:
# Get points of the line
mid_point1 = vtk.vtkVector3d(0,0,0)
mid_line.GetNthControlPointPosition(0,mid_point1)

mid_point2 = vtk.vtkVector3d(0,0,0)
mid_line.GetNthControlPointPosition(1,mid_point2)

# arc tangent in radians
angle = math.atan2(mid_point2[2] - mid_point1[2], mid_point2[0] - mid_point1[0])

# Center of cylinder
mid_center_cylinder = [mid_point1[0] + cylinder_radius * math.cos(angle) + np.sign(mid_point1[2] - mid_point2[2]) * np.sign(math.cos(angle)) * cylinder_height/2 * abs(math.sin(angle)),
                   mid_point1[1],
                   mid_point1[2] + cylinder_radius * math.sin(angle) + np.sign(math.cos(angle)) * cylinder_height/2 * math.cos(angle)]

# direction of cylinder
mid_cylinder_direction = [np.sign(mid_point1[2] - mid_point2[2]) * np.sign(math.cos(angle)) * abs(math.sin(angle)), 
                      0,
                      abs(math.cos(angle))]


## Find the center and direction of right cylinder  

In [10]:
# Get points of the line
right_point1 = vtk.vtkVector3d(0,0,0)
right_line.GetNthControlPointPosition(0,right_point1)

right_point2 = vtk.vtkVector3d(0,0,0)
right_line.GetNthControlPointPosition(1,right_point2)

# arc tangent in radians
angle = math.atan2(right_point2[2] - right_point1[2], right_point2[0] - right_point1[0])

# Center of cylinder
right_center_cylinder = [right_point1[0] + cylinder_radius * math.cos(angle) + np.sign(right_point1[2] - right_point2[2]) * np.sign(math.cos(angle)) * cylinder_height/2 * abs(math.sin(angle)),
                   right_point1[1],
                   right_point1[2] + cylinder_radius * math.sin(angle) + np.sign(math.cos(angle)) * cylinder_height/2 * math.cos(angle)]

# direction of cylinder
right_cylinder_direction = [np.sign(right_point1[2] - right_point2[2]) * np.sign(math.cos(angle)) * abs(math.sin(angle)), 
                      0,
                      abs(math.cos(angle))]


## Create 3x cylinders

In [11]:
# Left
left_cylinder = pv.Cylinder(center=left_center_cylinder, direction=left_cylinder_direction,
                            radius=cylinder_radius, height=cylinder_height)
left_cylinderNode = slicer.modules.models.logic().AddModel(left_cylinder)
left_cylinderNode.SetName('LeftCylinder') 


# Mid
mid_cylinder = pv.Cylinder(center=mid_center_cylinder, direction=mid_cylinder_direction,
                            radius=cylinder_radius, height=cylinder_height)
mid_cylinderNode = slicer.modules.models.logic().AddModel(mid_cylinder)
mid_cylinderNode.SetName('MidCylinder')


# Right
right_cylinder = pv.Cylinder(center=right_center_cylinder, direction=right_cylinder_direction,
                            radius=cylinder_radius, height=cylinder_height)
right_cylinderNode = slicer.modules.models.logic().AddModel(right_cylinder)
right_cylinderNode.SetName('RightCylinder')


## Convert the models to segmentations to visualize them

In [12]:
# Left
# Create a new segmentation node
segmentationNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLSegmentationNode")
segmentationNode.SetName("left_cylinder")
    
# Import model into segmentation node
slicer.modules.segmentations.logic().ImportModelToSegmentationNode(left_cylinderNode, segmentationNode)


# Mid
# Create a new segmentation node
segmentationNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLSegmentationNode")
segmentationNode.SetName("mid_cylinder")
    
# Import model into segmentation node
slicer.modules.segmentations.logic().ImportModelToSegmentationNode(mid_cylinderNode, segmentationNode)
    

# Right
# Create a new segmentation node
segmentationNode = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLSegmentationNode")
segmentationNode.SetName("right_cylinder")
    
# Import model into segmentation node
slicer.modules.segmentations.logic().ImportModelToSegmentationNode(right_cylinderNode, segmentationNode)
    


True

## Save variables

In [19]:
Specifications_file_path = str(parentpath) + '/L_Specifications.txt'

In [20]:
# Variables to save
Specifications_type = "L_specifications_low-quality"
outputVolumeSpacingMm = volume_node.GetSpacing()

# Combine variables into a matrix
specifications = {
    "Specifications": Specifications_type,
    "microCT_file": nrrd_file,
    "spacing_mm": outputVolumeSpacingMm,
    "output_volume_spacing_mm": outputVolumeSpacingMm,
    "cylinder_height": cylinder_height,
    "cylinder_radius": cylinder_radius,
    "left_cylinder_center": left_center_cylinder,
    "left_cylinder_direction": left_cylinder_direction,
    "mid_cylinder_center": mid_center_cylinder,
    "mid_cylinder_direction": mid_cylinder_direction,
    "right_cylinder_center": right_center_cylinder,
    "right_cylinder_direction": right_cylinder_direction,

}

# create (overwrite) the file in append mode
with open(Specifications_file_path, "w") as file:
    # Write each variable as "variable_name: variable"
    for name, value in specifications.items():
        file.write(f"{name}: {value}\n")


## Save the "L" line for ROI

In [21]:
#working directory
wd = os.getcwd()

In [22]:
# Left: Save a markup node as .mrk.json
markup_node = slicer.util.getNode('L')
markup_output_file = os.path.join(wd, parentpath, 'left_L.mrk.json')
slicer.util.saveNode(markup_node, markup_output_file)

# Mid: Save a markup node as .mrk.json
markup_node = slicer.util.getNode('L_1')
markup_output_file = os.path.join(wd, parentpath, 'mid_L.mrk.json')

slicer.util.saveNode(markup_node, markup_output_file)

# Right: Save a markup node as .mrk.json
markup_node = slicer.util.getNode('L_2')
markup_output_file = os.path.join(wd, parentpath, 'right_L.mrk.json')

slicer.util.saveNode(markup_node, markup_output_file)

True